# Lecture 11: Pivots and Joins
v.ekc

Two tools today: **pivot** turns two categorical columns into a grid of summaries, and **join** glues two tables together on a shared column. The methods are similar to `group`!

**Today's sections:**
1. Rows from lists
2. Review — grouping
3. Cross-classification
4. Pivot tables
5. Joins
6. Try It — Skyscrapers

In [1]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.filterwarnings("ignore")

---
## 1. Rows from Lists

So far we've built tables column by column. You can also start from the labels and add **rows** (as lists; remember, lists can mix types).

In [76]:
Table().with_columns('Numbers', np.arange(1, 4))

Numbers
1
2
3


In [82]:
drinks = Table(['Drink', 'Cafe', 'Price'])
drinks

Drink,Cafe,Price


In [83]:
drinks = drinks.with_rows([
    ['Matcha Rose Latte', 'Northtown', 6],
    ['Cosmic Chai', 'Northtown', 5.75],
    ['Espresso', 'Cafe Mokka',  2.75],
    ['Latte',    'Cafe Mokka',  4],
    ['Espresso', "Jitter Bean",   2.65]
])
drinks

Drink,Cafe,Price
Matcha Rose Latte,Northtown,6
Cosmic Chai,Northtown,5.75
Espresso,Cafe Mokka,2.75
Latte,Cafe Mokka,4
Espresso,Jitter Bean,2.65


### Try It 1: Your drink 😊

1. Add your own favorite drink as a new row (drink, cafe, price).

2. How many rows does your new table have?

In [79]:
# 1. add your drink



Drink,Cafe,Price
Matcha Latte,Northtown,5.5
Espresso,Cafe Mokka,2.75
Latte,Cafe Mokka,5.25
Espresso,Jitter Bean,2.0
Dubai latte,Stu Brew,8


In [80]:
# 2. row count



4

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`with_rows` takes a list of row-lists, and returns a new table.*

```python
# 1.
my_drinks = drinks.with_rows([['Honey Bee Latte', 'Northtown', 6]])
my_drinks

# 2.
my_drinks.num_rows
```

</details>

---
## 2. Review... Grouping

Warm up on the welcome survey: `group` with one column counts; with a function, it summarizes.

In [88]:
survey = Table.read_table('data111_survey_fa26.csv')
survey.show(3)

Timestamp,Pets,Handedness,Extroversion,Texts,Hours of Sleep,Sleep Position,Pant Leg,Piercings,Food Preference
8/24/2026 16:41:18,Cat,Right,3,1,8,On my left side,Right leg,2,Sweet
8/24/2026 21:40:00,Fish,Left,8,2,4.5,On my left side,Right leg,0,Savory
8/25/2026 9:31:37,"Dog, Cat, Bird, Other",Right,5,3,7.5,On my right side,I don't wear pants,0,Sweet


In [89]:
survey.group("Handedness")

Handedness,count
Left,2
Right,13


In [90]:
(survey
 .select('Handedness', 'Hours of Sleep')
 .group("Handedness", np.average))

Handedness,Hours of Sleep average
Left,5.75
Right,7.18462


### Try It 2: Group review 😊

1. For each handedness, what is the **longest** night of sleep reported? (`np.max` is a collect function too.)

In [ ]:
# 1. max sleep by handedness



<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Any summary function slots into `group`'s second argument.*

```python
# 1.
survey.select('Handedness', 'Hours of Sleep').group('Handedness', np.max)
```

</details>

---
## 3. Cross-Classification

Group by a *list* of two columns → one row per combination.

In [91]:
survey.group(["Handedness", 'Sleep Position'])

Handedness,Sleep Position,count
Left,On my left side,1
Left,On my right side,1
Right,On my back like a vampire,2
Right,On my left side,2
Right,On my right side,4
Right,On my stomach,5


In [92]:
(survey
 .select("Handedness", 'Sleep Position', "Hours of Sleep")
 .group(["Handedness", 'Sleep Position'], collect=np.average))

Handedness,Sleep Position,Hours of Sleep average
Left,On my left side,4.5
Left,On my right side,7
Right,On my back like a vampire,7.7
Right,On my left side,7.25
Right,On my right side,8
Right,On my stomach,6.3


### Try It 3: Cross-classified texts 😊

1. For every Handedness × Sleep Position combo, find the **average number of texts**.

2. Which combo texts the most, and how seriously would you take that finding? 😉

In [101]:
# 1. average texts per combo


Handedness,Sleep Position,Texts average
Left,On my left side,2
Left,On my right side,6
Right,On my back like a vampire,7.5
Right,On my left side,1.5
Right,On my right side,5
Right,On my stomach,4


*2. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same two-column group, just swap the value column.*

```python
# 1.
(survey
 .select('Handedness', 'Sleep Position', 'Texts')
 .group(['Handedness', 'Sleep Position'], collect=np.average))
```

*2. Read the largest average, but tiny combos (one or two students) make flashy averages that mean very little.*

</details>

---
## 4. Pivot Tables

A pivot shows the same information as a two-column group, but as a **grid**: one variable across the top, one down the side. Much easier to compare.

### 📋 Reference Table

| Code | What it does |
|---|---|
| `t.pivot('cols', 'rows')` | grid of **counts** |
| `t.pivot('cols', 'rows', values='v', collect=f)` | grid of `f` applied to `v` |


In [99]:
survey.pivot('Handedness', 'Sleep Position')

Sleep Position,Left,Right
On my back like a vampire,0,2
On my left side,1,2
On my right side,1,4
On my stomach,0,5


In [100]:
# t.pivot('col', 'row', values = 'col_to_aggregate', collect = func_name)

survey.pivot('Handedness',
             'Sleep Position', 
             values='Hours of Sleep', 
             collect=np.average)

Sleep Position,Left,Right
On my back like a vampire,0,7.7
On my left side,4.5,7.25
On my right side,7,8
On my stomach,0,6.3


In [98]:
# This cell will throw an error
# You cannot include just 1 of the optional arguments!
#    does not know what column to take the average of :(

survey.pivot('Sleep Position', 
             'Handedness', 
             collect=np.average)

TypeError: collect requires values to be specified

### Try It 4: Pivot practice 😊

1. Rebuild your Try It 3 result as a **pivot** grid: Sleep Position across the top, Handedness down the side, average `Texts` in the cells.

2. Which layout is easier to compare, the grouped rows or the grid?

In [ ]:
# 1. pivot with values and collect



*2. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`values` + `collect` travel together, that error above is why.*

```python
# 1.
survey.pivot('Sleep Position', 'Handedness', values='Texts', collect=np.average)
```

*2. Same numbers, but the grid puts every comparison side by side: that's the point of pivot.*

</details>

---
### Try It 6: Skyscrapers 🏙️

Run the next cell to load ~1,700 U.S. skyscrapers (name, material, city, height, year completed).

1. For each city, how tall is the tallest building for each material?

2. For each city, what's the height difference between the tallest steel building and the tallest concrete building?

3. Generate a table of the **names** of the oldest buildings for each material for each city. *Hint: sort first, then what does `pivot` collect?*

In [2]:
# From the CORGIS Dataset Project
# By Austin Cory Bart acbart@vt.edu
# Version 2.0.0, created 3/22/2016
# https://corgis-edu.github.io/corgis/csv/skyscrapers/

sky = Table.read_table('skyscrapers.csv')
sky = (sky.with_column('age', 2024 - sky.column('completed'))
          .drop('completed'))
sky.show(3)

name,material,city,height,age
One World Trade Center,mixed/composite,New York City,541.3,10
Willis Tower,steel,Chicago,442.14,50
432 Park Avenue,concrete,New York City,425.5,9


In [3]:
# 1. For each city, how tall is the tallest building for each material?



city,concrete,mixed/composite,steel
Atlanta,264.25,311.8,169.47
Austin,208.15,0,93.6
Baltimore,161.24,0,155.15
Boston,121.92,139,240.79
Charlotte,265.48,239.7,179.23
Chicago,423.22,306.94,442.14
Cincinnati,125,202.69,175
Cleveland,125,288.65,215.8
Columbus,79.25,0,169.3
Dallas,176.48,280.72,270.06


In [45]:
# 2. For each city, what’s the height difference between the tallest 
#    steel building and the tallest concrete building?
# Hint: you just found the tallest steel building and tallest concrete building for each city!



In [4]:
# 3. Generate a pivot table of the names of the oldest buildings for each 
#    material for each city:

# Hint: You can use sort to find the name of the oldest building in the dataset
#        Sort the table first and then pivot it!
sky.sort('age', descending=True).column('name').item(0)

# you can use this function to return the first item on a list!
def first(s):
    return s.item(0)



<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*All three are pivots, the only choices are `values` and `collect`.*

```python
# 1.
tallest = sky.pivot('material', 'city', values='height', collect=max)
tallest

# 2.
tallest.with_column(
    'difference',
    abs(tallest.column('steel') - tallest.column('concrete')))

# 3.
def first(s):
    return s.item(0)

sky.sort('age').pivot('material', 'city', values='name', collect=first)
```

</details>

---
> **Today's takeaway:** group counts, pivot grids, join glues. We'll practice all three on a big real dataset next lecture.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — pivot & join
```python
t.pivot('col_across', 'col_down', values='v', collect=np.average)
a.join('shared_col', b, 'its_name_in_b')
```

---
## 5. Joins

When two tables share a column, `join` matches their rows up.

### 📋 Reference Table

| Code | What it does |
|---|---|
| `a.join('col_a', b, 'col_b')` | keep rows where the values match |
| unmatched rows | quietly dropped! |
| repeated values | every matching pair appears |

In [104]:
drinks

Drink,Cafe,Price
Matcha Rose Latte,Northtown,6
Cosmic Chai,Northtown,5.75
Espresso,Cafe Mokka,2.75
Latte,Cafe Mokka,4
Espresso,Jitter Bean,2.65


In [105]:
discounts = Table().with_columns(
    'Coupon % off', make_array(10, 25, 5),
    'Location', make_array('Northtown', 'Cafe Mokka', 'Northtown')
)
discounts

Coupon % off,Location
10,Northtown
25,Cafe Mokka
5,Northtown


Join on the shared column, every drink picks up its cafe's coupon:

In [106]:
combined = drinks.join('Cafe', discounts, 'Location')
combined

Cafe,Drink,Price,Coupon % off
Cafe Mokka,Espresso,2.75,25
Cafe Mokka,Latte,4,25
Northtown,Matcha Rose Latte,6,10
Northtown,Matcha Rose Latte,6,5
Northtown,Cosmic Chai,5.75,10
Northtown,Cosmic Chai,5.75,5


In [107]:
discounted_frac = 1 - combined.column('Coupon % off') / 100
combined.with_column(
    'Discounted Price', 
    combined.column('Price') * discounted_frac
)

Cafe,Drink,Price,Coupon % off,Discounted Price
Cafe Mokka,Espresso,2.75,25,2.0625
Cafe Mokka,Latte,4,25,3
Northtown,Matcha Rose Latte,6,10,5.4
Northtown,Matcha Rose Latte,6,5,5.7
Northtown,Cosmic Chai,5.75,10,5.175
Northtown,Cosmic Chai,5.75,5,5.4625


One more twist: join a table with *itself*:

In [ ]:
drinks.join('Cafe', drinks, 'Cafe')

> **Look closely at that last one**: joining `drinks` with *itself* pairs every drink with every other drink at the same cafe. Joins multiply matching rows; always sanity-check `num_rows` after a join.

### Try It 5: Rate the cafes 😊

Run the next cell to get a table of cafe ratings.

1. Join `drinks` with `ratings` so every drink shows its cafe's stars.

2. What's the **cheapest** drink at the top-rated cafe? (Two sorts, or a `where` + a sort.)

In [ ]:
# run me: cafe ratings
ratings = Table().with_columns(
    'Cafe', make_array('Northtown', 'Cafe Mokka', 'Jitter Bean'),
    'Stars', make_array(4.5, 4.8, 4.2))
ratings

In [ ]:
# 1. join drinks with ratings



In [ ]:
# 2. cheapest drink at the best cafe
#     hint: filter the table to the BEST cafe


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Join on the shared Cafe column; then it's ordinary sorting.*

```python
# 1.
rated = drinks.join('Cafe', ratings, 'Cafe')
rated

# 2.
rated.sort('Stars', descending=True).sort('Price')   # eyeball it, or:
rated.where('Stars', max(ratings.column('Stars'))).sort('Price').column('Drink').item(0)
```

</details>